# 05 · Model Comparison

Compare the baseline and gradient-boosting model using validation choices and an untouched final month.

## Reading guide

This notebook is part of a connected workflow. It states the decision being made, shows the supporting checks and records limitations alongside the result. Source files are never modified in place.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = Path(os.environ.get("FIFAR_DATA_DIR", PROJECT_ROOT / "data" / "raw" / "FiFAR"))
REPORTS = PROJECT_ROOT / "reports"
IMAGES = PROJECT_ROOT / "images"

sns.set_theme(style="whitegrid")
CORAL = "#F08FA0"
TEAL = "#0E6268"
DARK = "#15262B"

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        "Set FIFAR_DATA_DIR to the extracted official FiFAR directory before running this notebook."
    )

In [ ]:
metrics = json.loads((REPORTS / 'model_metrics.json').read_text())

In [ ]:
rows = []
for model_name, model_result in metrics["models"].items():
    for split in ["validation", "test"]:
        rows.append({"model": model_name, "split": split, **model_result[split]})
comparison = pd.DataFrame(rows)
comparison

## 1. Ranking performance

In [ ]:
comparison.pivot(index='model', columns='split', values=['average_precision', 'roc_auc'])

Average precision is the primary ranking metric because it reflects performance on the rare positive class. ROC AUC is reported as a secondary measure.

## 2. Review-capacity comparison

In [ ]:
capacity = []
for model_name, result in metrics["models"].items():
    for row in result["test_capacity"]:
        capacity.append({"model": model_name, **row})
capacity = pd.DataFrame(capacity)
capacity

In [ ]:
plt.figure(figsize=(10, 6))
for model_name, group in capacity.groupby("model"):
    plt.plot(group.review_share * 100, group.recall_at_capacity * 100, marker="o", linewidth=2.5, label=model_name.replace("_", " ").title())
plt.xlabel("Applications reviewed (%)")
plt.ylabel("Fraud recovered (%)")
plt.title("Final-month recovery by review capacity")
plt.legend(frameon=False)
plt.show()

## 3. Selection decision

In [ ]:
three_percent = capacity[capacity.review_share.eq(.03)].set_index("model")
three_percent[["fraud_captured", "precision_at_capacity", "recall_at_capacity"]]

Histogram gradient boosting is selected because it improves validation average precision and fraud recovery at every tested capacity. Its final-month results are reported once, after that selection.

## Limitation

The comparison covers two well-defined baselines, not an exhaustive model search. Additional models should be added only if they improve the validation result and preserve reproducibility.